In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate() 

In [2]:
spark

In [6]:
df = spark.read.option("header", True).option("inferSchema", True).csv("salesdata/*.csv")

In [7]:
df.count()

186850

In [8]:
df.printSchema()

root
 |-- Order ID: integer (nullable = true)
 |-- Product: string (nullable = true)
 |-- Quantity Ordered: integer (nullable = true)
 |-- Price Each: double (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Purchase Address: string (nullable = true)



In [9]:
ventas = df.withColumnRenamed("Order ID", "OrderID").withColumnRenamed("Quantity Ordered", "Quantity").withColumnRenamed("Price Each", "Price")
ventas = ventas.withColumnRenamed("Order Date", "OrderDate").withColumnRenamed("Purchase Address", "OrderAddress")

In [10]:
# Comprobamos si tenemos campos nulos
from pyspark.sql.functions import col
ventas.filter(col("OrderID").isNull()).show()

+-------+-------+--------+-----+----------+----------------+
|OrderID|Product|Quantity|Price| OrderDate|    OrderAddress|
+-------+-------+--------+-----+----------+----------------+
|   NULL|Product|    NULL| NULL|Order Date|Purchase Address|
|   NULL|   NULL|    NULL| NULL|      NULL|            NULL|
|   NULL|   NULL|    NULL| NULL|      NULL|            NULL|
|   NULL|   NULL|    NULL| NULL|      NULL|            NULL|
|   NULL|Product|    NULL| NULL|Order Date|Purchase Address|
|   NULL|Product|    NULL| NULL|Order Date|Purchase Address|
|   NULL|   NULL|    NULL| NULL|      NULL|            NULL|
|   NULL|   NULL|    NULL| NULL|      NULL|            NULL|
|   NULL|   NULL|    NULL| NULL|      NULL|            NULL|
|   NULL|   NULL|    NULL| NULL|      NULL|            NULL|
|   NULL|Product|    NULL| NULL|Order Date|Purchase Address|
|   NULL|   NULL|    NULL| NULL|      NULL|            NULL|
|   NULL|   NULL|    NULL| NULL|      NULL|            NULL|
|   NULL|   NULL|    NUL

In [11]:
# Quitar nulos
ventasSinNulos = ventas.na.drop("any")
# Comprobamos
ventasSinNulos.filter(col("OrderID").isNull()).show()

+-------+-------+--------+-----+---------+------------+
|OrderID|Product|Quantity|Price|OrderDate|OrderAddress|
+-------+-------+--------+-----+---------+------------+
+-------+-------+--------+-----+---------+------------+



In [12]:
# Comprobamos las cabeceras
encabezados = ventasSinNulos.filter((col("OrderID") == "Order ID") | (col("Product") == "Product"))
encabezados.show()

+-------+-------+--------+-----+---------+------------+
|OrderID|Product|Quantity|Price|OrderDate|OrderAddress|
+-------+-------+--------+-----+---------+------------+
+-------+-------+--------+-----+---------+------------+



In [13]:
from pyspark.sql.functions import split, trim
ventasCiudad = ventasSinNulos.withColumn("City", trim(split(col("OrderAddress"), ",")[1]))
ventasEstado = ventasCiudad.withColumn("State", trim(split(split(col("OrderAddress"), ",")[2], " ")[1]))
ventasEstado.show()


+-------+--------------------+--------+------+--------------+--------------------+-------------+-----+
|OrderID|             Product|Quantity| Price|     OrderDate|        OrderAddress|         City|State|
+-------+--------------------+--------+------+--------------+--------------------+-------------+-----+
| 295665|  Macbook Pro Laptop|       1|1700.0|12/30/19 00:01|136 Church St, Ne...|New York City|   NY|
| 295666|  LG Washing Machine|       1| 600.0|12/29/19 07:03|562 2nd St, New Y...|New York City|   NY|
| 295667|USB-C Charging Cable|       1| 11.95|12/12/19 18:21|277 Main St, New ...|New York City|   NY|
| 295668|    27in FHD Monitor|       1|149.99|12/22/19 15:13|410 6th St, San F...|San Francisco|   CA|
| 295669|USB-C Charging Cable|       1| 11.95|12/18/19 12:38|43 Hill St, Atlan...|      Atlanta|   GA|
| 295670|AA Batteries (4-p...|       1|  3.84|12/31/19 22:58|200 Jefferson St,...|New York City|   NY|
| 295671|USB-C Charging Cable|       1| 11.95|12/16/19 15:10|928 12th St,

In [16]:
ventasEstado.select("City").distinct().show()

+-------------+
|         City|
+-------------+
|       Dallas|
|  Los Angeles|
|San Francisco|
|     Portland|
|       Austin|
|      Atlanta|
|      Seattle|
|New York City|
|       Boston|
+-------------+

